In [ ]:
# ===== SEL IDENTITAS - JANGAN DIHAPUS, HARUS SEL PERTAMA =====
import sys, platform, hashlib, datetime, zoneinfo

NAMA = 'Wandi'
NIM  = '244107020003'

N     = int(NIM[-3:])
PARAM = {
    'bins' : 2 ** ((N % 4) + 3),
    'clip' : round(1.0 + (N % 5) * 0.5, 1),
    'tile' : (N % 4) + 2,
    'L'    : 2 ** ((N % 3) + 1),
}
waktu = datetime.datetime.now(zoneinfo.ZoneInfo('Asia/Jakarta'))

print('NAMA   :', NAMA)
print('NIM    :', NIM, '| N =', N)
for k, v in PARAM.items():
    print('%-6s :' % k, v)
print('WAKTU  :', waktu.strftime('%Y-%m-%d %H:%M:%S %Z'))
print('SISTEM :', sys.version.split()[0], platform.platform())
kunci = NIM + '|' + waktu.strftime('%Y-%m-%d')
print('TOKEN  :', hashlib.sha256(kunci.encode()).hexdigest()[:16])

# Modul 5 - Histogram, Histogram Equalization, dan Dithering

Isi notebook: percobaan E-1 (histogram), E-2 (histogram equalization), E-3 (dithering),
beserta jawaban semua pertanyaan praktikumnya.

Tiap percobaan dikerjakan dua tahap: tahap 1 pakai citra contoh (lena.jpg / lena_lc.jpg) buat
mastiin kodenya sudah benar, tahap 2 diulang pakai citra KTM hasil akuisisi Pertemuan 2.

## Persiapan

Mount Drive dan import library.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import cv2 as cv
from google.colab.patches import cv2_imshow
from skimage import io
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import glob

In [ ]:
CONTOH = '/content/drive/MyDrive/PCVK/Images'          # citra contoh bersama
BASE   = '/content/drive/MyDrive/PCVK/Images/' + NIM   # citra KTM milik saya

print('isi folder contoh :', sorted(os.listdir(CONTOH)))
print('isi folder KTM    :', sorted(os.listdir(BASE)))

Catatan privasi sesuai ketentuan modul: berkas KTM asli cuma saya simpan di Google Drive yang
tidak dibagikan ke publik. Yang masuk ke repository cuma notebook-nya, dan keluaran citra KTM
saya bersihkan dulu sebelum di-push kalau repository-nya dibuat publik.

Satu fungsi bantu buat nampilin citra, biar judulnya otomatis kebawa NIM seperti yang diminta
di ketentuan pengumpulan.

In [ ]:
def tampil(img, judul, cmap=None):
    if img.ndim == 2:
        plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    else:
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    plt.title(NIM + ' - ' + judul, fontsize=9)
    plt.axis('off')

# E-1 PERCOBAAN HISTOGRAM

## Tahap 1 - histogram manual pada citra contoh lena.jpg

In [ ]:
# membuat histogram image (manual)
img = cv.imread(CONTOH + '/lena.jpg')
img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

height, width, depth = np.shape(img)
names = np.arange(256)

red   = [0]*256
green = [0]*256
blue  = [0]*256

for y in range(0, height):
    for x in range(0, width):
        red[img[y][x][0]]   += 1
        green[img[y][x][1]] += 1
        blue[img[y][x][2]]  += 1

fig, axs = plt.subplots(1, 3, figsize=[20,5], sharex=True, sharey=True)
fig.suptitle(NIM + ' - Histogram RGB plot lena.jpg')
fig.text(0.09, 0.5, 'Jumlah Kemunculan', va='center', rotation='vertical')
fig.text(0.5, 0.04, 'Intensitas Warna', ha='center')
axs[0].bar(names, red, color='red')
axs[1].bar(names, green, color='green')
axs[2].bar(names, blue, color='blue')
plt.show()

Bentuknya sudah sama dengan gambar contoh di modul, jadi kodenya saya pakai lagi untuk citra
KTM. Biar tidak nulis ulang perulangannya, saya bungkus jadi fungsi dulu.

In [ ]:
def histogram_manual(img, L=256):
    h = np.zeros(L, dtype=np.int64)     # 1. siapkan penghitung, isi 0
    for y in range(img.shape[0]):       # 2. telusuri setiap baris
        for x in range(img.shape[1]):   #    dan setiap kolom
            h[img[y, x]] += 1           # 3. tambah 1 sesuai nilai piksel
    return h

gray = cv.imread(BASE + '/KTM1b.jpg', cv.IMREAD_GRAYSCALE)
h    = histogram_manual(gray)
p    = h / gray.size                    # 4. normalisasi, jumlahnya = 1
cdf  = np.cumsum(p)                     # 5. distribusi kumulatif
assert h.sum() == gray.size             #    pemeriksaan wajib

print('jumlah piksel   :', gray.size)
print('jumlah histogram:', h.sum())
print('p total         : %.6f' % p.sum())
print('cdf terakhir    : %.6f' % cdf[-1])

## Tahap 2 - histogram citra KTM sendiri (KTM1b.jpg)

In [ ]:
ktm1b = cv.imread(BASE + '/KTM1b.jpg')

plt.figure(figsize=(14,4))
plt.subplot(1,2,1)
tampil(ktm1b, 'KTM1b.jpg asli')
plt.subplot(1,2,2)
for i, (nama, warna) in enumerate([('Biru','blue'), ('Hijau','green'), ('Merah','red')]):
    plt.plot(histogram_manual(ktm1b[:, :, i]), color=warna, label=nama)
plt.title(NIM + ' - histogram manual RGB KTM1b', fontsize=9)
plt.xlabel('intensitas warna')
plt.ylabel('jumlah kemunculan')
plt.legend()
plt.show()

## PERTANYAAN PRAKTIKUM E1

### 1. Bandingkan histogram manual, numpy.histogram, dan cv.calcHist

Ketiganya harusnya menghasilkan angka yang sama persis, jadi selisih maksimumnya harus nol.
Dicoba dulu di lena.jpg, terus diulang di KTM1b.jpg.

In [ ]:
def banding_tiga_cara(gray, nama_berkas):
    manual = histogram_manual(gray)
    numpy_h, _ = np.histogram(gray, bins=256, range=(0,256))
    calc   = cv.calcHist([gray], [0], None, [256], [0,256]).ravel().astype(np.int64)

    print(nama_berkas)
    print('  manual vs numpy   : selisih maksimum =', int(np.abs(manual - numpy_h).max()))
    print('  manual vs calcHist: selisih maksimum =', int(np.abs(manual - calc).max()))
    print('  numpy  vs calcHist: selisih maksimum =', int(np.abs(numpy_h - calc).max()))
    print('  total ketiganya   :', manual.sum(), numpy_h.sum(), int(calc.sum()))
    return manual, numpy_h, calc

lena_gray = cv.imread(CONTOH + '/lena.jpg', cv.IMREAD_GRAYSCALE)
banding_tiga_cara(lena_gray, 'lena.jpg')
m, n_, c_ = banding_tiga_cara(gray, 'KTM1b.jpg')

In [ ]:
# sekalian dibandingkan waktunya, soalnya perulangan bersarang itu lambat
import time

t0 = time.time(); histogram_manual(gray);                      t_manual = time.time() - t0
t0 = time.time(); np.histogram(gray, bins=256, range=(0,256)); t_numpy  = time.time() - t0
t0 = time.time(); cv.calcHist([gray], [0], None, [256], [0,256]); t_calc = time.time() - t0

print('manual   : %.4f detik' % t_manual)
print('numpy    : %.4f detik' % t_numpy)
print('calcHist : %.4f detik' % t_calc)

Selisih maksimum ketiganya nol, baik di lena.jpg maupun di KTM1b.jpg, dan jumlah
totalnya sama dengan jumlah piksel citra. Wajar, karena ketiganya sebenarnya melakukan hal yang
sama persis: menghitung berapa kali tiap nilai 0-255 muncul. Yang beda cuma cara jalannya,
perulangan bersarang di Python dijalankan satu per satu di interpreter, sedangkan numpy dan
calcHist jalan di kode C, makanya jauh lebih cepat (lihat perbandingan waktunya di atas).
Kesimpulannya versi manual dipakai buat memahami alurnya, tapi kalau citranya besar mending
pakai np.histogram atau cv.calcHist.

### 2. Histogram ternormalisasi p(j) dan kurva CDF untuk KTM1a sampai KTM1d

In [ ]:
berkas = ['KTM1a.jpg', 'KTM1b.jpg', 'KTM1c.jpg', 'KTM1d.jpg']
kondisi = ['ruangan gelap', 'lampu ruangan', 'pakai flash', 'cahaya matahari']

fig, axs = plt.subplots(2, 2, figsize=(13,7))
statistik = []
for ax, nama, kond in zip(axs.ravel(), berkas, kondisi):
    g = cv.imread(BASE + '/' + nama, cv.IMREAD_GRAYSCALE)
    h = histogram_manual(g)
    p = h / g.size
    cdf = np.cumsum(p)

    ax.bar(np.arange(256), p, color='gray', width=1.0)
    ax2 = ax.twinx()
    ax2.plot(cdf, color='red')
    ax2.set_ylim(0, 1.05)
    ax.set_title(NIM + ' - ' + nama + ' (' + kond + ')', fontsize=9)
    ax.set_xlabel('intensitas')
    ax.set_ylabel('p(j)')
    ax2.set_ylabel('CDF')

    # cari intensitas di mana cdf menyentuh 0.5, itu nilai tengah sebarannya
    median = int(np.searchsorted(cdf, 0.5))
    statistik.append((nama, kond, g.mean(), g.std(), median, int(g.min()), int(g.max())))
plt.tight_layout()
plt.show()

print('%-11s %-17s %7s %7s %8s %6s %6s' % ('berkas','kondisi','mean','std','median','min','max'))
for s in statistik:
    print('%-11s %-17s %7.1f %7.1f %8d %6d %6d' % s)

Jawaban nomor 2. Bentuk kurva CDF-nya beda jelas antara citra paling gelap dan paling terang.

Pada KTM1a (difoto di ruangan gelap), p(j)-nya numpuk di sisi kiri, jadi kurva CDF-nya naik
curam di intensitas rendah terus langsung mendatar sebelum sampai tengah. Artinya hampir semua
piksel nilainya kecil dan hampir tidak ada piksel terang sama sekali. Kebalikannya pada citra
paling terang (KTM1d yang kena cahaya matahari, atau KTM1c yang pakai flash), CDF-nya baru mulai
naik setelah intensitas menengah dan naiknya curam di bagian kanan, karena pikselnya numpuk di
nilai besar. Kolom median di tabel juga menunjukkan hal yang sama: makin terang kondisi
pencahayaannya, makin besar intensitas tempat CDF menyentuh 0.5.

Kalau dikaitkan sama akuisisi di Pertemuan 2, ini masuk akal. Foto di ruangan gelap cahaya yang
sampai ke sensor sedikit, jadi rentang nilai yang kepakai cuma sebagian kecil di ujung gelap.
Foto yang kena flash atau matahari langsung kebalikannya, bahkan sebagian bisa mentok di 255 dan
detailnya hilang (kelihatan dari nilai max yang menempel di 255 dan lonjakan p(j) di ujung kanan).
Yang pencahayaannya paling wajar adalah yang pakai lampu ruangan, CDF-nya naik paling landai dan
merata dari kiri ke kanan, artinya rentang intensitasnya kepakai paling banyak.

### 3. Histogram KTM1b dengan PARAM['bins'] bin dan dengan 256 bin

In [ ]:
bins = PARAM['bins']
h_kecil, tepi = np.histogram(gray, bins=bins, range=(0,256))
h_penuh, _    = np.histogram(gray, bins=256, range=(0,256))

plt.figure(figsize=(13,4))
plt.subplot(1,2,1)
plt.bar(tepi[:-1], h_kecil, width=256/bins, align='edge', edgecolor='k')
plt.title(NIM + ' - KTM1b, ' + str(bins) + ' bin', fontsize=9)
plt.xlabel('intensitas'); plt.ylabel('jumlah piksel')
plt.subplot(1,2,2)
plt.bar(np.arange(256), h_penuh, width=1.0)
plt.title(NIM + ' - KTM1b, 256 bin', fontsize=9)
plt.xlabel('intensitas')
plt.show()

print('lebar tiap bin versi', bins, 'bin :', 256//bins, 'tingkat intensitas')
print('jumlah total kedua versi sama?', h_kecil.sum() == h_penuh.sum(), '(', h_kecil.sum(), ')')
print('jumlah puncak lokal 256 bin :', int(((h_penuh[1:-1] > h_penuh[:-2]) & (h_penuh[1:-1] > h_penuh[2:])).sum()))
print('jumlah puncak lokal', bins, 'bin :', int(((h_kecil[1:-1] > h_kecil[:-2]) & (h_kecil[1:-1] > h_kecil[2:])).sum()))

Jumlah total pikselnya tetap sama, yang berubah cuma cara mengelompokkannya. Dengan
PARAM['bins'] bin, satu batang mewakili beberapa tingkat intensitas sekaligus, jadi semua variasi
di dalam satu kelompok itu dilebur jadi satu angka.

Yang hilang adalah detail bentuk sebarannya: puncak-puncak sempit yang di versi 256 bin
kelihatan terpisah (lihat jumlah puncak lokalnya jauh berkurang) jadi menyatu jadi satu batang
lebar. Padahal pada citra KTM, puncak-puncak sempit itu justru penting, karena biasanya mewakili
warna dasar kartu, warna tulisan, dan warna foto yang nilainya berdekatan. Kalau bin-nya sedikit,
kita cuma bisa bilang "citranya cenderung gelap atau terang", tapi tidak bisa lagi lihat ada
berapa kelompok intensitas dan di mana persisnya celah kosong antar kelompok. Jadi bin sedikit
enak buat lihat gambaran kasar, bin 256 buat analisa detail.

# E-2 PERCOBAAN HISTOGRAM EQUALIZATION

## Tahap 1 - ekualisasi manual pada citra contoh lena_lc.jpg

Rumus yang dipakai yang di ulasan teori:

Ko = round( Ci * (2^k - 1) / (w * h) )

Ci distribusi kumulatif nilai piksel, w lebar citra, h tinggi citra, k jumlah bit (8).

In [ ]:
def he_manual(gray, k=8):
    h   = histogram_manual(gray)          # frekuensi tiap nilai
    ci  = np.cumsum(h)                    # distribusi kumulatif
    w   = gray.shape[1]
    tinggi = gray.shape[0]
    lut = np.round(ci * (2**k - 1) / (w * tinggi)).astype(np.uint8)   # rumus Ko
    return lut[gray], lut

lena_lc = cv.imread(CONTOH + '/lena_lc.jpg')
b, g, r = cv.split(lena_lc)
he_lc = cv.merge([he_manual(b)[0], he_manual(g)[0], he_manual(r)[0]])

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); tampil(lena_lc, 'lena_lc.jpg sebelum')
plt.subplot(1,2,2); tampil(he_lc, 'lena_lc.jpg sesudah HE manual')
plt.show()

In [ ]:
# histogram sebelum dan sesudah, biar kelihatan sebarannya melebar
plt.figure(figsize=(13,5))
for kolom, (citra, judul) in enumerate([(lena_lc, 'sebelum'), (he_lc, 'sesudah')]):
    for i, warna in enumerate(['blue','green','red']):
        plt.subplot(2, 3, kolom*3 + i + 1)
        plt.hist(citra[:, :, i].ravel(), bins=256, range=(0,256), color=warna)
        plt.title(NIM + ' - ' + judul + ' ' + warna, fontsize=8)
        plt.yticks([])
plt.tight_layout()
plt.show()

Hasilnya sudah sama dengan gambar contoh di modul (yang kiri pucat, yang kanan kontrasnya naik),
jadi lanjut ke tahap 2 pakai KTM1a.jpg, yaitu foto yang diambil di ruangan gelap.

In [ ]:
ktm1a = cv.imread(BASE + '/KTM1a.jpg')
b, g, r = cv.split(ktm1a)
he_ktm1a = cv.merge([he_manual(b)[0], he_manual(g)[0], he_manual(r)[0]])

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); tampil(ktm1a, 'KTM1a.jpg sebelum')
plt.subplot(1,2,2); tampil(he_ktm1a, 'KTM1a.jpg sesudah HE manual')
plt.show()

## 2. Bandingkan hasil manual dengan cv.equalizeHist

In [ ]:
def banding_he(bgr, nama_berkas):
    b, g, r = cv.split(bgr)

    b_equalized = cv.equalizeHist(b)
    g_equalized = cv.equalizeHist(g)
    r_equalized = cv.equalizeHist(r)
    he_cv = cv.merge([b_equalized, g_equalized, r_equalized])

    he_saya = cv.merge([he_manual(b)[0], he_manual(g)[0], he_manual(r)[0]])

    selisih = np.abs(he_cv.astype(int) - he_saya.astype(int))
    print(nama_berkas, '-> selisih maksimum =', int(selisih.max()),
          ', rata-rata = %.4f' % selisih.mean(),
          ', piksel yang beda = %.2f%%' % (100 * (selisih > 0).mean()))
    return he_cv, he_saya

cv_lc, saya_lc = banding_he(lena_lc, 'lena_lc.jpg')
cv_1a, saya_1a = banding_he(ktm1a, 'KTM1a.jpg')

In [ ]:
# cari tahu asal selisihnya: bandingkan tabel LUT tiap kanal
for i, nama_kanal in enumerate(['Biru', 'Hijau', 'Merah']):
    kanal = cv.split(ktm1a)[i]
    lut_saya = he_manual(kanal)[1]
    eq = cv.equalizeHist(kanal)

    lut_cv = np.zeros(256, np.uint8)
    dipakai = np.zeros(256, bool)
    for v in range(256):
        pas = kanal == v
        if pas.any():
            lut_cv[v] = eq[pas][0]
            dipakai[v] = True

    beda = [v for v in range(256) if dipakai[v] and int(lut_saya[v]) != int(lut_cv[v])]
    h = histogram_manual(kanal)
    pertama = int(np.argmax(h > 0))
    print('kanal %-6s: nilai yang LUT-nya beda = %3d dari %3d nilai yang dipakai'
          % (nama_kanal, len(beda), int(dipakai.sum())))
    print('              bin tidak nol pertama = %d, isinya %d piksel (%.2f%% dari total)'
          % (pertama, int(h[pertama]), 100 * h[pertama] / kanal.size))
    for v in beda[:5]:
        print('              %3d -> saya %3d, OpenCV %3d' % (v, lut_saya[v], lut_cv[v]))

Pada citra contoh lena_lc.jpg selisihnya nol, tapi pada citra KTM saya keluar
selisih kecil, beberapa tingkat intensitas saja. Penyebabnya ada dua dan dua-duanya soal cara
menghitung, bukan salah kode:

1. Rumus di modul membagi distribusi kumulatif dengan seluruh jumlah piksel (w x h), sedangkan
   cv.equalizeHist mengurangi dulu jumlah piksel pada bin tidak nol yang pertama, baru
   dibagi. Jadi penyebutnya beda sedikit dan seluruh tabel LUT-nya ikut bergeser.
2. Sisanya soal pembulatan. Nilai hasil rumus itu pecahan, dan pembulatan ke integer terdekat
   pada nilai yang kebetulan jatuh persis di tengah bisa beda arah antara np.round dan
   pembulatan yang dipakai OpenCV. Bedanya paling banyak satu tingkat intensitas.

Cetakan per kanal di atas menunjukkan hal ini: kanal yang bin pertamanya cuma berisi belasan
piksel LUT-nya sama persis, sedangkan kanal yang bin pertamanya berisi ratusan piksel LUT-nya
bergeser di banyak nilai. Karena selisihnya cuma 1-2 tingkat, secara visual dua hasil itu tetap
tidak bisa dibedakan mata.

## 3. Ekualisasi kanal B, G, R satu per satu

In [ ]:
berkas = CONTOH + '/lena.jpg'          # tahap 1; tahap 2 ganti: BASE + '/KTM1d.jpg'
bgr = cv.imread(berkas)

# cara 1: tiap kanal diekualisasi sendiri-sendiri
kanal = list(cv.split(bgr))
for i in range(3):
    kanal[i] = cv.equalizeHist(kanal[i])
he_rgb = cv.merge(kanal)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); tampil(bgr, 'lena.jpg asli')
plt.subplot(1,2,2); tampil(he_rgb, 'lena.jpg ekualisasi B, G, R')
plt.show()

In [ ]:
# tahap 2, diulang pada KTM1d.jpg punya saya
berkas = BASE + '/KTM1d.jpg'
bgr = cv.imread(berkas)

kanal = list(cv.split(bgr))
for i in range(3):
    kanal[i] = cv.equalizeHist(kanal[i])
he_rgb = cv.merge(kanal)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); tampil(bgr, 'KTM1d.jpg asli')
plt.subplot(1,2,2); tampil(he_rgb, 'KTM1d.jpg ekualisasi B, G, R')
plt.show()

Warnanya jadi berubah, tidak cuma tambah kontras. Masuk akal, soalnya tiap kanal diregangkan
sendiri-sendiri dengan tabel LUT yang beda, jadi perbandingan R : G : B pada satu piksel ikut
berubah. Padahal perbandingan itulah yang menentukan warnanya.

## 4. Cara kedua: pisahkan kecerahan dari informasi warna

In [ ]:
# cara 2: hanya kanal terang yang diekualisasi
ycrcb    = cv.cvtColor(bgr, cv.COLOR_BGR2YCrCb)
y, cr, cb = cv.split(ycrcb)
he_y = cv.cvtColor(cv.merge([cv.equalizeHist(y), cr, cb]), cv.COLOR_YCrCb2BGR)

# varian setara pada ruang warna lain
hsv        = cv.cvtColor(bgr, cv.COLOR_BGR2HSV)
h_, sat, v = cv.split(hsv)
he_v = cv.cvtColor(cv.merge([h_, sat, cv.equalizeHist(v)]), cv.COLOR_HSV2BGR)

lab      = cv.cvtColor(bgr, cv.COLOR_BGR2LAB)
l, a, b_ = cv.split(lab)
he_l = cv.cvtColor(cv.merge([cv.equalizeHist(l), a, b_]), cv.COLOR_LAB2BGR)

In [ ]:
judul = ['Asli', 'Ekualisasi B, G, R', 'Kanal Y', 'Kanal V', 'Kanal L']
citra = [bgr, he_rgb, he_y, he_v, he_l]

plt.figure(figsize=(18, 4))
for i, (c, t) in enumerate(zip(citra, judul), 1):
    plt.subplot(1, 5, i)
    plt.imshow(cv.cvtColor(c, cv.COLOR_BGR2RGB))
    plt.title(NIM + ' - ' + t, fontsize=9)
    plt.axis('off')
plt.show()

In [ ]:
def geser_hue(asli, hasil):
    h1 = cv.cvtColor(asli,  cv.COLOR_BGR2HSV)[:, :, 0].astype(np.int16)
    h2 = cv.cvtColor(hasil, cv.COLOR_BGR2HSV)[:, :, 0].astype(np.int16)
    d  = np.abs(h1 - h2)
    d  = np.minimum(d, 180 - d)          # Hue melingkar 0-179
    return d.mean()

print('%-22s %16s %16s' % ('Cara', 'geser Hue (der)', 'rerata terang'))
for t, c in zip(judul, citra):
    print('%-22s %16.2f %16.1f' % (t, geser_hue(bgr, c) * 2,
                                   cv.cvtColor(c, cv.COLOR_BGR2GRAY).mean()))

### Tabel hasil

| Cara ekualisasi | Rata-rata pergeseran Hue (derajat) | Rerata kecerahan sesudah | Catatan pengamatan visual |
|---|---|---|---|
| Citra asli | - | lihat cetakan di atas | warna asli KTM, gelap atau terang sesuai kondisi akuisisi |
| Ekualisasi kanal B, G, R | paling besar dari keempatnya | bergeser ke tengah | warna kartu ikut berubah, muncul semburat warna yang tidak ada di aslinya |
| Kanal Y (YCrCb) | kecil | bergeser ke tengah | kontras naik, warna kartu praktis tetap |
| Kanal V (HSV) | kecil | bergeser ke tengah | mirip Y, warna sedikit lebih pekat karena saturation tidak ikut disesuaikan |
| Kanal L (LAB) | kecil | bergeser ke tengah | perpindahan terang-gelapnya paling rata |

Angka persisnya ada di cetakan tabel di atas, karena beda-beda tiap citra KTM. Kolom kecerahan
saya tulis "bergeser ke tengah" karena arahnya tergantung citranya: kalau citranya gelap
reratanya naik, kalau citranya sudah terang seperti KTM1d reratanya malah turun. Memang itu
tujuan ekualisasi, menyebarkan nilai ke seluruh rentang, bukan menaikkan kecerahan.

### Jawaban pertanyaan nomor 4

**1. Cara mana yang menghasilkan pergeseran Hue paling besar?**

Ekualisasi kanal B, G, R satu per satu. Angkanya ada di baris kedua tabel cetakan di atas, dan
nilainya jauh lebih besar dibanding tiga cara lainnya. Sebabnya tiap kanal punya histogram dan
LUT sendiri, jadi setelah diekualisasi perbandingan R : G : B pada satu piksel berubah. Hue
dihitung dari perbandingan itu, makanya warnanya ikut geser.

**2. Kenapa ekualisasi kanal terang tetap tidak menghasilkan pergeseran Hue persis nol?**

Secara teori kanal Cr, Cb (atau H, S, atau a, b) tidak disentuh sama sekali, jadi harusnya Hue
tetap. Tapi nilainya tidak nol karena citra harus dikembalikan ke ruang BGR, dan di situ ada dua
hal: hasil perkalian matriks konversi dibulatkan ke integer, lalu nilai yang keluar dari rentang
0-255 dipotong (truncate). Dua hal ini terjadi per piksel, dan piksel yang setelah ekualisasi
nilainya mepet 0 atau 255 paling sering kena potong. Setelah dibalik lagi ke HSV untuk diukur,
perbandingan kanalnya sudah bergeser sedikit, jadi muncul selisih Hue yang kecil tapi bukan nol.

**3. Kanal mana yang paling sesuai untuk foto KTM saya?**

Saya pilih kanal L (LAB). Bagian citra yang saya jadikan dasar penilaian adalah blok teks NIM
dan nama, bukan bagian foto wajah, karena itu yang harus kebaca.

Kalau dilihat dari kolom pergeseran Hue saja, justru Y dan V yang angkanya paling kecil, jadi
dari sisi kesetiaan warna dua kanal itu sedikit lebih aman. Tapi L dirancang mengikuti persepsi
terang mata manusia, jadi peregangan kontrasnya terasa paling rata dan batas antara tinta gelap
dengan dasar kartu paling tegas, sedangkan pergeseran Hue-nya masih di kisaran yang sama kecilnya
dengan Y dan V, jauh di bawah cara B, G, R. Kanal V hasilnya mirip tapi karena V cuma nilai
maksimum dari R, G, B, bagian kartu yang warnanya pekat jadi terasa berlebihan. Jadi selisih Hue
yang sedikit lebih besar itu saya anggap harga yang pantas untuk teks yang lebih terbaca.

**4. Ganti cv.equalizeHist dengan CLAHE pada kanal terang**

In [ ]:
clahe = cv.createCLAHE(clipLimit=PARAM['clip'], tileGridSize=(PARAM['tile'], PARAM['tile']))

lab_c = cv.cvtColor(bgr, cv.COLOR_BGR2LAB)
l, a, b_ = cv.split(lab_c)
clahe_l = cv.cvtColor(cv.merge([clahe.apply(l), a, b_]), cv.COLOR_LAB2BGR)

plt.figure(figsize=(16,4.5))
plt.subplot(1,3,1); tampil(bgr, 'asli')
plt.subplot(1,3,2); tampil(he_l, 'kanal L, equalizeHist')
plt.subplot(1,3,3); tampil(clahe_l, 'kanal L, CLAHE clip ' + str(PARAM['clip']))
plt.show()

print('%-28s %16s %16s' % ('Cara', 'geser Hue (der)', 'rerata terang'))
for t, c in [('equalizeHist kanal L', he_l), ('CLAHE kanal L', clahe_l)]:
    print('%-28s %16.2f %16.1f' % (t, geser_hue(bgr, c) * 2,
                                   cv.cvtColor(c, cv.COLOR_BGR2GRAY).mean()))

CLAHE pergeseran Hue-nya lebih kecil dan rerata kecerahannya tidak melonjak sejauh equalizeHist.
Masuk akal, karena equalizeHist meregangkan histogram seluruh citra sekaligus sampai mentok,
sedangkan CLAHE membagi citra jadi kotak-kotak kecil dan kenaikan kontrasnya dibatasi clipLimit.
Piksel yang nilainya jadi mepet 0 atau 255 lebih sedikit, jadi yang kena potong waktu balik ke
BGR juga lebih sedikit.

## PERTANYAAN PRAKTIKUM E2

### 1. Ekualisasi global pada citra KTM (KTM1a.jpg versi grayscale)

In [ ]:
g1a = cv.imread(BASE + '/KTM1a.jpg', cv.IMREAD_GRAYSCALE)
he_saya_1a = he_manual(g1a)[0]
he_cv_1a   = cv.equalizeHist(g1a)

print('selisih maksimum manual vs equalizeHist :', int(np.abs(he_saya_1a.astype(int) - he_cv_1a.astype(int)).max()))

plt.figure(figsize=(15,7))
plt.subplot(2,3,1); tampil(g1a, 'KTM1a asli (grayscale)')
plt.subplot(2,3,2); tampil(he_saya_1a, 'HE manual')
plt.subplot(2,3,3); tampil(he_cv_1a, 'cv.equalizeHist')
for i, (citra, judul) in enumerate([(g1a,'asli'), (he_saya_1a,'HE manual'), (he_cv_1a,'equalizeHist')]):
    plt.subplot(2,3,4+i)
    plt.hist(citra.ravel(), bins=256, range=(0,256), color='gray')
    plt.title(NIM + ' - histogram ' + judul, fontsize=9)
    plt.yticks([])
plt.tight_layout()
plt.show()

In [ ]:
def PSNR(img1, img2):
    mse = np.mean((img1.astype(float) - img2.astype(float)) ** 2)
    if mse == 0:
        return 100
    return 20 * math.log10(255.0 / math.sqrt(mse))

print('PSNR asli vs HE manual    : %.2f dB' % PSNR(g1a, he_saya_1a))
print('PSNR asli vs equalizeHist : %.2f dB' % PSNR(g1a, he_cv_1a))
print('rerata terang  : asli %.1f -> hasil %.1f' % (g1a.mean(), he_cv_1a.mean()))
print('std (kontras)  : asli %.1f -> hasil %.1f' % (g1a.std(), he_cv_1a.std()))

Jawaban 1.c. PSNR-nya kecil, jauh di bawah citra hasil denoising yang biasanya di atas 30 dB,
padahal citranya jelas lebih kebaca. Sebabnya PSNR itu dihitung dari MSE, yaitu rata-rata
kuadrat selisih nilai piksel terhadap citra acuan. Histogram equalization memang kerjanya
mengubah hampir semua nilai piksel, piksel yang tadinya numpuk di nilai 40-60 bisa dilempar ke
120-200. Selisihnya besar, MSE-nya besar, jadi PSNR-nya turun. Penurunan PSNR di sini justru
tanda operasinya bekerja, bukan tanda citranya rusak.

Kesimpulannya PSNR tidak layak dipakai menilai keterbacaan citra. PSNR mengukur "seberapa mirip
dengan citra aslinya", sedangkan yang kita kejar di sini malah sengaja bikin citranya beda dari
aslinya supaya kontrasnya naik. PSNR cocok untuk kasus di mana citra asli memang acuan yang
benar, misalnya menilai hasil denoising atau kompresi. Untuk keterbacaan, ukuran yang lebih masuk
akal adalah kontras (std), sebaran histogram, atau langsung dicoba dibaca tulisannya.

### 2. Ekualisasi lokal dengan CLAHE pada citra KTM

In [ ]:
clahe = cv.createCLAHE(clipLimit=PARAM['clip'], tileGridSize=(PARAM['tile'], PARAM['tile']))
clahe_1a = clahe.apply(g1a)

plt.figure(figsize=(16,4.5))
plt.subplot(1,3,1); tampil(g1a, 'KTM1a asli')
plt.subplot(1,3,2); tampil(he_cv_1a, 'ekualisasi global')
plt.subplot(1,3,3); tampil(clahe_1a, 'CLAHE clip ' + str(PARAM['clip']) + ' tile ' + str(PARAM['tile']))
plt.show()

print('%-20s %8s %8s %12s' % ('cara', 'mean', 'std', 'lap var'))
for nama, c in [('asli', g1a), ('global', he_cv_1a), ('CLAHE', clahe_1a)]:
    print('%-20s %8.1f %8.1f %12.1f' % (nama, c.mean(), c.std(), cv.Laplacian(c, cv.CV_64F).var()))

Jawaban 2.a. NIM dan nama pada kartu lebih terbaca pada hasil CLAHE. Ekualisasi global
memakai satu tabel LUT untuk seluruh citra, jadi kalau pencahayaan foto tidak rata (satu sisi
kartu lebih gelap karena bayangan tangan atau lampu dari satu arah), sisi yang gelap tetap
tertinggal gelap sementara sisi yang terang malah jadi mentok putih. CLAHE menghitung
ekualisasi per kotak kecil, jadi tiap bagian kartu dapat peregangan kontras sesuai kondisi
lokalnya. Angka varians Laplacian di atas juga menunjukkan tepi tulisan pada hasil CLAHE lebih
tajam.

**2.b. Coba tiga nilai clipLimit lain di sekitar nilai saya**

In [ ]:
nilai_clip = [PARAM['clip'], PARAM['clip'] + 1, PARAM['clip'] + 2, PARAM['clip'] + 4]

plt.figure(figsize=(17,4.5))
for i, cl in enumerate(nilai_clip, 1):
    c = cv.createCLAHE(clipLimit=cl, tileGridSize=(PARAM['tile'], PARAM['tile'])).apply(g1a)
    plt.subplot(1, 4, i)
    tampil(c, 'clip ' + str(cl))
plt.show()

print('%8s %8s %12s %18s' % ('clip', 'std', 'lap var', 'std area rata'))
for cl in nilai_clip:
    c = cv.createCLAHE(clipLimit=cl, tileGridSize=(PARAM['tile'], PARAM['tile'])).apply(g1a)
    # ambil sepotong area yang warnanya rata (pojok kanan bawah kartu) buat lihat noise
    hh, ww = c.shape
    polos = c[int(0.75*hh):int(0.95*hh), int(0.70*ww):int(0.95*ww)]
    print('%8.1f %8.1f %12.1f %18.2f' % (cl, c.std(), cv.Laplacian(c, cv.CV_64F).var(), polos.std()))

In [ ]:
# bagian yang noise-nya paling kelihatan waktu clipLimit dinaikkan
hh, ww = g1a.shape
potong = (slice(int(0.75*hh), int(0.95*hh)), slice(int(0.70*ww), int(0.95*ww)))

plt.figure(figsize=(16,4))
for i, cl in enumerate(nilai_clip, 1):
    c = cv.createCLAHE(clipLimit=cl, tileGridSize=(PARAM['tile'], PARAM['tile'])).apply(g1a)
    plt.subplot(1, 4, i)
    tampil(c[potong], 'area polos, clip ' + str(cl))
plt.show()

Jawaban 2.b. Makin besar clipLimit, kontras memang makin naik (kolom std dan varians
Laplacian naik terus), tapi kolom terakhir menunjukkan std pada area kartu yang warnanya
seharusnya rata juga ikut naik. Itu bukan detail, itu noise yang ikut diperkuat. Pada potongan
area polos di atas kelihatan mulai muncul bintik-bintik kasar dan batas kotak antar tile begitu
clipLimit dinaikkan.

Nilai terbaik menurut saya adalah clipLimit yang masih menaikkan ketajaman tulisan tapi belum
bikin area polos berbintik, yaitu nilai PARAM['clip'] sampai satu tingkat di atasnya. Di atas itu
tulisan tidak jadi lebih terbaca, yang bertambah cuma noise-nya. Ini sekaligus alasan kenapa
CLAHE pakai batas clipLimit: tanpa batas, ekualisasi pada kotak yang isinya hampir seragam akan
meregangkan beda nilai yang sangat kecil (yang isinya cuma noise) jadi beda yang besar.

# E-3 TUGAS PRAKTIKUM DITHERING

## 1. Dithering Floyd-Steinberg

Kernel yang dipakai:

    (1/16) [ -    #    7 ]
           [ 3    5    1 ]

Konvensi yang dipakai di seluruh modul: x kolom, y baris, pemindaian dari kiri ke kanan lalu
baris atas ke bawah. Jadi bobot 7/16 untuk tetangga kanan, 3/16 untuk baris bawah satu kolom
sebelumnya, 5/16 tepat di bawah, dan 1/16 baris bawah satu kolom setelahnya.

In [ ]:
def floyd_steinberg(gray, L=2):
    img  = gray.astype(np.float32).copy()
    step = 255.0 / (L - 1)
    H, W = img.shape
    for y in range(H):
        for x in range(W):
            lama  = img[y, x]
            baru  = np.round(lama / step) * step      # warna palet terdekat
            img[y, x] = baru
            error = lama - baru
            if x + 1 < W:                img[y,   x+1] += error * 7 / 16
            if x > 0     and y + 1 < H:  img[y+1, x-1] += error * 3 / 16
            if y + 1 < H:                img[y+1, x  ] += error * 5 / 16
            if x + 1 < W and y + 1 < H:  img[y+1, x+1] += error * 1 / 16
    return np.clip(img, 0, 255).astype(np.uint8)

In [ ]:
# tahap 1, citra contoh lena.jpg dengan 2 level
lena_gray = cv.imread(CONTOH + '/lena.jpg', cv.IMREAD_GRAYSCALE)
dither_lena = floyd_steinberg(lena_gray, L=2)

plt.figure(figsize=(12,6))
plt.subplot(1,2,1); tampil(lena_gray, 'lena.jpg asli')
plt.subplot(1,2,2); tampil(dither_lena, 'Floyd-Steinberg, L = 2')
plt.show()

print('jumlah warna berbeda sesudah dithering :', len(np.unique(dither_lena)))

In [ ]:
# tahap 2, citra KTM sendiri dengan jumlah level dari NIM
g1b = cv.imread(BASE + '/KTM1b.jpg', cv.IMREAD_GRAYSCALE)
dither_ktm = floyd_steinberg(g1b, L=PARAM['L'])

plt.figure(figsize=(12,6))
plt.subplot(1,2,1); tampil(g1b, 'KTM1b.jpg asli')
plt.subplot(1,2,2); tampil(dither_ktm, 'Floyd-Steinberg, L = ' + str(PARAM['L']))
plt.show()

print('L =', PARAM['L'], '-> jumlah warna berbeda :', len(np.unique(dither_ktm)))
print('nilai warna yang dipakai :', np.unique(dither_ktm))

In [ ]:
# pembanding: kuantisasi biasa tanpa error diffusion, jumlah level sama
step = 255.0 / (PARAM['L'] - 1)
kuantisasi = (np.round(g1b.astype(np.float32) / step) * step).astype(np.uint8)

plt.figure(figsize=(16,5))
plt.subplot(1,3,1); tampil(g1b, 'KTM1b asli')
plt.subplot(1,3,2); tampil(kuantisasi, 'kuantisasi biasa, L = ' + str(PARAM['L']))
plt.subplot(1,3,3); tampil(dither_ktm, 'dithering, L = ' + str(PARAM['L']))
plt.show()

Dua-duanya cuma memakai jumlah warna yang sama, tapi kuantisasi biasa hasilnya blok-blok besar
dan gradasinya hilang, sedangkan yang pakai error diffusion gradasinya masih kelihatan dari
jauh karena selisih warnanya disebar ke tetangga jadi pola titik.

## Menghitung satu langkah error diffusion secara manual

Biar yakin implementasinya sudah benar, saya hitung satu langkah pertama dengan tangan pada
potongan 3 x 3 berikut, pakai L = 2 sehingga step = 255.

    100  200   50
     80  150  220
     30   90  170

Piksel pertama yang diproses adalah (y=0, x=0) bernilai 100.

- baru = round(100 / 255) x 255 = round(0.392) x 255 = 0 x 255 = 0
- error = 100 - 0 = 100
- tetangga kanan (0,1): 200 + 100 x 7/16 = 200 + 43.75 = 243.75
- tetangga kiri bawah (1,-1): di luar citra, dilewati
- tetangga bawah (1,0): 80 + 100 x 5/16 = 80 + 31.25 = 111.25
- tetangga kanan bawah (1,1): 150 + 100 x 1/16 = 150 + 6.25 = 156.25

Jadi setelah satu langkah, isi citranya harus jadi:

      0   243.75    50
    111.25 156.25  220
     30     90     170

In [ ]:
patch = np.array([[100, 200,  50],
                  [ 80, 150, 220],
                  [ 30,  90, 170]], dtype=np.uint8)

# satu langkah saja, ditulis lepas tanpa perulangan biar bisa dicocokkan dengan hitungan tangan
img  = patch.astype(np.float32).copy()
step = 255.0 / (2 - 1)
lama = img[0, 0]
baru = np.round(lama / step) * step
img[0, 0] = baru
error = lama - baru
img[0, 1] += error * 7 / 16
img[1, 0] += error * 5 / 16
img[1, 1] += error * 1 / 16

print('lama  =', lama)
print('baru  =', baru)
print('error =', error)
print('hasil program setelah 1 langkah:')
print(img)

manual = np.array([[  0.00, 243.75,  50.00],
                   [111.25, 156.25, 220.00],
                   [ 30.00,  90.00, 170.00]], dtype=np.float32)
print('cocok dengan hitungan tangan?', np.allclose(img, manual))

Hasil programnya sama persis dengan hitungan tangan, jadi urutan bobot 7/16, 3/16, 5/16, 1/16
dan arah tetangganya sudah benar.

## 2. Grayscale, ekualisasi, baru dithering

Dibandingkan juga dengan dithering langsung tanpa ekualisasi, untuk lihat urutan mana yang bikin
tulisan pada KTM lebih terbaca.

In [ ]:
# tahap 1, citra contoh lena_lc.jpg
lc_gray = cv.imread(CONTOH + '/lena_lc.jpg', cv.IMREAD_GRAYSCALE)
lc_dither_langsung = floyd_steinberg(lc_gray, L=2)
lc_he = cv.equalizeHist(lc_gray)
lc_he_dither = floyd_steinberg(lc_he, L=2)

plt.figure(figsize=(18,5))
plt.subplot(1,4,1); tampil(lc_gray, 'lena_lc.jpg asli')
plt.subplot(1,4,2); tampil(lc_dither_langsung, 'dithering langsung')
plt.subplot(1,4,3); tampil(lc_he, 'hasil ekualisasi')
plt.subplot(1,4,4); tampil(lc_he_dither, 'ekualisasi lalu dithering')
plt.show()

In [ ]:
# tahap 2, citra KTM1a.jpg punya saya
dither_langsung = floyd_steinberg(g1a, L=PARAM['L'])
he_dulu         = cv.equalizeHist(g1a)
he_lalu_dither  = floyd_steinberg(he_dulu, L=PARAM['L'])

plt.figure(figsize=(18,5))
plt.subplot(1,4,1); tampil(g1a, 'KTM1a.jpg asli')
plt.subplot(1,4,2); tampil(dither_langsung, 'dithering langsung')
plt.subplot(1,4,3); tampil(he_dulu, 'hasil ekualisasi')
plt.subplot(1,4,4); tampil(he_lalu_dither, 'ekualisasi lalu dithering')
plt.show()

print('%-26s %8s %8s %12s' % ('cara', 'mean', 'std', 'lap var'))
for nama, c in [('asli', g1a), ('dithering langsung', dither_langsung),
                ('ekualisasi lalu dithering', he_lalu_dither)]:
    print('%-26s %8.1f %8.1f %12.1f' % (nama, c.mean(), c.std(), cv.Laplacian(c, cv.CV_64F).var()))

In [ ]:
# diperbesar pada area tulisan supaya kebacanya bisa dinilai
hh, ww = g1a.shape
teks = (slice(int(0.45*hh), int(0.75*hh)), slice(int(0.30*ww), int(0.95*ww)))

plt.figure(figsize=(16,5))
plt.subplot(1,3,1); tampil(g1a[teks], 'area teks - asli')
plt.subplot(1,3,2); tampil(dither_langsung[teks], 'area teks - dithering langsung')
plt.subplot(1,3,3); tampil(he_lalu_dither[teks], 'area teks - ekualisasi lalu dithering')
plt.show()

Menurut saya urutan yang bikin teks pada KTM lebih terbaca adalah ekualisasi dulu, baru
dithering.

Alasannya, KTM1a difoto di ruangan gelap, jadi seluruh nilai pikselnya numpuk di rentang sempit
di ujung gelap. Kalau langsung didithering, semua nilai yang berdekatan itu dibandingkan dengan
palet warna yang jaraknya jauh (step = 255 / (L-1)), dan hampir semuanya dibulatkan ke level
yang sama. Hasilnya bagian tulisan dan bagian dasar kartu jatuh ke warna yang sama, sehingga
tulisan hilang, cuma tersisa pola titik acak dari error yang terdifusi.

Kalau diekualisasi dulu, selisih kecil antara tinta dan dasar kartu diregangkan dulu jadi
selisih yang besar. Waktu dithering, dua bagian itu sudah jatuh ke level palet yang berbeda,
jadi bentuk hurufnya tetap kelihatan. Angka varians Laplacian pada cetakan di atas juga
menunjukkan hasil urutan ini tepinya paling tegas.

Catatan tambahan, di potongan area teks kelihatan efek sampingnya: ekualisasi ikut mengangkat
noise di bagian dasar kartu, jadi latar belakangnya jadi lebih berbintik dibanding dithering
langsung. Tapi untuk tujuan membaca NIM dan nama, tepi huruf yang tegas jauh lebih penting
daripada latar yang bersih.

# Kesimpulan

Histogram itu cuma hitungan berapa kali tiap nilai intensitas muncul, dan mau dihitung manual,
pakai np.histogram, atau cv.calcHist hasilnya sama persis, yang beda cuma kecepatannya. Bentuk
histogram dan kurva CDF-nya ternyata bisa dipakai membaca kondisi pencahayaan waktu foto diambil:
citra dari ruangan gelap CDF-nya naik curam di kiri, citra kena matahari naik curam di kanan.

Histogram equalization meratakan sebaran itu dengan memakai CDF sebagai fungsi transformasi.
Hasilnya jauh lebih kebaca, tapi PSNR-nya malah turun, karena PSNR mengukur kemiripan dengan
citra asli sedangkan di sini kita memang sengaja mengubah nilai pikselnya. Kalau ekualisasi
dikenakan ke kanal B, G, R sendiri-sendiri warnanya ikut geser, jadi lebih aman mengerjakan di
kanal terang saja (Y, V, atau L). CLAHE lebih cocok lagi untuk foto KTM karena pencahayaannya
tidak rata, cuma clipLimit-nya tidak boleh terlalu besar supaya noise tidak ikut diperkuat.

Dithering Floyd-Steinberg mengurangi jumlah warna tapi tetap mempertahankan kesan gradasi, dengan
cara menyebarkan error kuantisasi ke tetangga yang belum diproses memakai bobot 7/16, 3/16, 5/16,
dan 1/16. Satu langkah pertamanya sudah saya cocokkan dengan hitungan tangan dan hasilnya sama.
Untuk citra yang kontrasnya rendah, dithering harus didahului ekualisasi, kalau tidak tulisan dan
latarnya jatuh ke level palet yang sama dan hurufnya hilang.